In [1]:
import pandas as pd 
from collections import defaultdict

In [2]:
matches = pd.read_csv('../data/processed/features_basic.csv') 
matches['MatchDateTime'] = pd.to_datetime(matches['MatchDateTime']) 
matches = matches.sort_values('MatchDateTime').reset_index(drop=True)

In [5]:
home_games_played = []
away_games_played = []

home_points = []
away_points = []

home_ppg = []
away_ppg = []

home_goals_pg = []
away_goals_pg = []

home_goals_against_pg = []
away_goals_against_pg = []

home_gd_pg = []
away_gd_pg = []

In [4]:
season_history = defaultdict(lambda: defaultdict(list))

In [6]:
def calculate_season_stats(previous_matches, team):
    if len(previous_matches) == 0:
        return {
            'games': 0,
            'points': 0,
            'ppg': 0,
            'goals_pg': 0,
            'goals_against_pg': 0,
            'gd_pg': 0
        }

    games = len(previous_matches)
    points = 0
    goals_for = 0
    goals_against = 0

    for match in previous_matches:

        # Team was home
        if match['HomeTeam'] == team:
            goals_for += match['FTHG']
            goals_against += match['FTAG']

            if match['FTR'] == 'H':
                points += 3
            elif match['FTR'] == 'D':
                points += 1

        # Team was away
        else:
            goals_for += match['FTAG']
            goals_against += match['FTHG']

            if match['FTR'] == 'A':
                points += 3
            elif match['FTR'] == 'D':
                points += 1

    return {
        'games': games,
        'points': points,
        'ppg': points / games,
        'goals_pg': goals_for / games,
        'goals_against_pg': goals_against / games,
        'gd_pg': (goals_for - goals_against) / games
    }

In [7]:
for _, current_match in matches.iterrows():

    season = current_match['Season']
    home_team = current_match['HomeTeam']
    away_team = current_match['AwayTeam']

    # Previous matches in the SAME season
    home_previous = season_history[season][home_team]
    away_previous = season_history[season][away_team]

    # Calculate stats BEFORE current match
    home_stats = calculate_season_stats(home_previous, home_team)
    away_stats = calculate_season_stats(away_previous, away_team)

    # Store home features
    home_games_played.append(home_stats['games'])
    home_points.append(home_stats['points'])
    home_ppg.append(home_stats['ppg'])
    home_goals_pg.append(home_stats['goals_pg'])
    home_goals_against_pg.append(home_stats['goals_against_pg'])
    home_gd_pg.append(home_stats['gd_pg'])

    # Store away features
    away_games_played.append(away_stats['games'])
    away_points.append(away_stats['points'])
    away_ppg.append(away_stats['ppg'])
    away_goals_pg.append(away_stats['goals_pg'])
    away_goals_against_pg.append(away_stats['goals_against_pg'])
    away_gd_pg.append(away_stats['gd_pg'])

    # Add current match to history AFTER calculating features
    season_history[season][home_team].append(current_match)
    season_history[season][away_team].append(current_match)

In [9]:
matches['HomeGamesPlayed'] = home_games_played
matches['AwayGamesPlayed'] = away_games_played

matches['HomePointsSeason'] = home_points
matches['AwayPointsSeason'] = away_points

matches['HomePPG'] = home_ppg
matches['AwayPPG'] = away_ppg

matches['HomeGoalsPerGame'] = home_goals_pg
matches['AwayGoalsPerGame'] = away_goals_pg

matches['HomeGoalsAgainstPerGame'] = home_goals_against_pg
matches['AwayGoalsAgainstPerGame'] = away_goals_against_pg

matches['HomeGDPerGame'] = home_gd_pg
matches['AwayGDPerGame'] = away_gd_pg

C:\Users\harry\AppData\Local\Temp\ipykernel_14276\1750599243.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['HomeGamesPlayed'] = home_games_played
C:\Users\harry\AppData\Local\Temp\ipykernel_14276\1750599243.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['AwayGamesPlayed'] = away_games_played
C:\Users\harry\AppData\Local\Temp\ipykernel_14276\1750599243.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  

In [10]:
matches['PPGDiff'] = matches['HomePPG'] - matches['AwayPPG']

matches['GDPerGameDiff'] = (
    matches['HomeGDPerGame'] - matches['AwayGDPerGame']
)

matches['GoalsPerGameDiff'] = (
    matches['HomeGoalsPerGame'] - matches['AwayGoalsPerGame']
)

# Positive means home team has the better defence
matches['GoalsAgainstPerGameDiff'] = (
    matches['AwayGoalsAgainstPerGame'] -
    matches['HomeGoalsAgainstPerGame']
)

C:\Users\harry\AppData\Local\Temp\ipykernel_14276\3582192768.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['PPGDiff'] = matches['HomePPG'] - matches['AwayPPG']
C:\Users\harry\AppData\Local\Temp\ipykernel_14276\3582192768.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches['GDPerGameDiff'] = (
C:\Users\harry\AppData\Local\Temp\ipykernel_14276\3582192768.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

In [11]:
matches[[
    'Season',
    'HomeTeam',
    'AwayTeam',
    'HomePPG',
    'AwayPPG',
    'PPGDiff',
    'HomeGDPerGame',
    'AwayGDPerGame',
    'GDPerGameDiff'
]].tail(10)

,Season,HomeTeam,AwayTeam,HomePPG,AwayPPG,PPGDiff,HomeGDPerGame,AwayGDPerGame,GDPerGameDiff
1890,25-26,Burnley,Wolves,0.567568,0.513514,0.054054,-1.000000,-1.108108,0.108108
1891,25-26,Crystal Palace,Arsenal,1.216216,2.216216,-1.000000,-0.243243,1.162162,-1.405405
1892,25-26,Brighton,Man United,1.432432,1.837838,-0.405405,0.243243,0.432432,-0.189189
1893,25-26,Tottenham,Everton,1.027027,1.324324,-0.297297,-0.270270,-0.054054,-0.216216
1894,25-26,Fulham,Newcastle,1.324324,1.324324,0.000000,-0.162162,0.000000,-0.162162
1895,25-26,Sunderland,Chelsea,1.378378,1.405405,-0.027027,-0.189189,0.189189,-0.378378
1896,25-26,Man City,Aston Villa,2.108108,1.675676,0.432432,1.162162,0.162162,1.000000
1897,25-26,Nott'm Forest,Bournemouth,1.162162,1.513514,-0.351351,-0.081081,0.108108,-0.189189
1898,25-26,Liverpool,Brentford,1.594595,1.405405,0.189189,0.270270,0.081081,0.189189
1899,25-26,West Ham,Leeds,0.972973,1.270270,-0.297297,-0.594595,-0.108108,-0.486486


In [12]:
print(matches['PPGDiff'].describe())
print()
print(matches['GDPerGameDiff'].describe())

count    1900.000000
mean       -0.028979
std         0.819496
min        -3.000000
25%        -0.563179
50%         0.000000
75%         0.500000
max         3.000000
Name: PPGDiff, dtype: float64

count    1900.000000
mean       -0.035307
std         1.258874
min        -6.000000
25%        -0.801231
50%         0.000000
75%         0.744643
max         4.500000
Name: GDPerGameDiff, dtype: float64


In [13]:
matches.to_csv('../data/processed/features_v2.csv', index=False)

print('Saved to data/processed/features_v2.csv')

Saved to data/processed/features_v2.csv
